In [7]:
# SETUP: SparkSession con Delta Lake
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, date_format, desc, to_timestamp, when
from pyspark.sql.types import *

spark = (
    SparkSession.builder.master("local[1]")
    .appName("DataExploration")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print("Delta Lake habilitado")

Spark version: 3.5.6
Delta Lake habilitado


In [8]:
DATA_DIR = "data"

apfel_df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(f"{DATA_DIR}/apfel_subscriptions.csv")
)
fenster_df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(f"{DATA_DIR}/fenster_subscriptions.csv")
)
exchange_df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(f"{DATA_DIR}/exchange_rates.csv")
)

print(f"Apfel:    {apfel_df.count()} filas, {len(apfel_df.columns)} columnas")
print(f"Fenster:  {fenster_df.count()} filas, {len(fenster_df.columns)} columnas")
print(f"Exchange: {exchange_df.count()} filas, {len(exchange_df.columns)} columnas")

Apfel:    1460 filas, 22 columnas
Fenster:  1165 filas, 26 columnas
Exchange: 13 filas, 3 columnas


In [31]:
apfel_df.printSchema()
apfel_df.limit(100).toPandas()

root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- customer_uuid: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_created_at: timestamp (nullable = true)
 |-- country_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- renewal_period: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- discount_code: string (nullable = true)
 |-- affiliate_id: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- app_version: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- internal_ref: string (nullable = true)
 |-- processing_status: string (nullable = true)



,event_id,event_timestamp,event_type,customer_uuid,customer_email,customer_created_at,country_code,region,city,postal_code,...,currency,amount,tax_amount,discount_code,affiliate_id,device_type,app_version,session_id,internal_ref,processing_status
0,APF-2025-000023,2025-02-01 10:56:16,SUBSCRIPTION_STARTED,APF_C00023,user23@test.com,2025-02-01 10:56:16,GB,Wales,Cardiff,CF10,...,EUR,59.99,12.0,SAVE20,None,android,4.1.5,sess_aebd07,REF-INT-023,completed
1,APF-2025-000063,2025-02-01 17:58:39,SUBSCRIPTION_STARTED,APF_C00055,user55@email.com,2025-02-01 17:58:39,DE,Saxony,Dresden,01067,...,EUR,59.99,11.4,None,None,ios,4.3.1,sess_6a9d56,REF-INT-063,completed
2,APF-2025-000031,2025-02-01 18:19:48,SUBSCRIPTION_STARTED,APF_C00031,user31@email.com,2025-02-01 18:19:48,GB,England,London,SW1A,...,EUR,119.99,24.0,NEWYEAR25,AFF002,web,4.1.3,sess_676675,REF-INT-031,completed
3,APF-2025-000095,2025-02-01 19:55:30,SUBSCRIPTION_STARTED,APF_C00075,user75@email.com,2025-02-01 19:55:30,GB,Scotland,Edinburgh,EH1,...,EUR,5.99,1.2,NEWYEAR25,AFF003,web,4.2.1,sess_d7fa9e,REF-INT-095,completed
4,APF-2025-000097,2025-02-02 13:56:02,SUBSCRIPTION_STARTED,APF_C00076,user76@email.com,2025-02-02 13:56:02,GB,England,London,SW1A,...,EUR,59.99,12.0,NEWYEAR25,AFF001,android,4.1.2,sess_5c9e24,REF-INT-097,completed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,APF-2025-000038,2025-02-26 15:21:17,SUBSCRIPTION_STARTED,APF_C00038,user38@email.com,2025-02-26 15:21:17,GB,England,London,SW1A,...,EUR,12.99,2.6,FLASH15,AFF002,android,4.1.0,sess_92b682,REF-INT-038,completed
96,APF-2025-000062,2025-02-26 19:00:16,SUBSCRIPTION_RENEWED,APF_C00031,user31@email.com,2025-02-01 18:19:48,GB,Scotland,Edinburgh,EH1,...,EUR,119.99,24.0,None,None,web,4.1.3,sess_f8bb85,REF-INT-062,completed
97,APF-2025-000051,2025-02-26 23:17:52,SUBSCRIPTION_STARTED,APF_C00051,user51@email.com,2025-02-26 23:17:52,GB,Scotland,Edinburgh,EH1,...,EUR,5.99,1.2,WELCOME10,AFF001,ios,4.3.1,sess_585baf,REF-INT-051,completed
98,APF-2025-000028,2025-02-26 23:29:03,SUBSCRIPTION_STARTED,APF_C00028,user28@email.com,2025-02-26 23:29:03,DE,Saxony,Dresden,01067,...,EUR,59.99,11.4,SAVE20,AFF002,ios,4.3.3,sess_b0ed11,REF-INT-028,completed


In [10]:
fenster_df.printSchema()
fenster_df.limit(100).toPandas()

root
 |-- id: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- type: string (nullable = true)
 |-- cid: string (nullable = true)
 |-- mail: string (nullable = true)
 |-- signup_ts: timestamp (nullable = true)
 |-- ctry: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- plan: string (nullable = true)
 |-- ccy: string (nullable = true)
 |-- price: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- vat_id: string (nullable = true)
 |-- campaign_src: string (nullable = true)
 |-- utm_medium: string (nullable = true)
 |-- utm_campaign: string (nullable = true)
 |-- browser: string (nullable = true)
 |-- os: string (nullable = true)
 |-- screen_res: string (nullable = true)
 |-- lang: string (nullable = true)
 |-- tz: string (nullable = true)
 |-- legacy_flag: boolean (nullable = true)
 |-- migrated_from: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- row_hash: string (nullable = true)


26/03/13 11:58:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,id,ts,type,cid,mail,signup_ts,ctry,state,zip,plan,...,utm_campaign,browser,os,screen_res,lang,tz,legacy_flag,migrated_from,batch_id,row_hash
0,FEN-00024,2025-02-01 13:38:59,new,FEN_C00024,user24@email.com,2025-02-01 13:38:59,USA,Washington,98101,premium_monthly,...,none,firefox,linux,1170x2532,es-MX,AEST,False,None,B001,fb743bc9
1,FEN-00033,2025-02-01 14:21:10,new,FEN_C00033,user33@email.com,2025-02-01 14:21:10,GBR,Scotland,EH1,standard_yearly,...,winter_sale,edge,windows,2560x1440,en-AU,GMT,False,None,B001,38071775
2,FEN-00065,2025-02-01 16:37:53,renew,FEN_C00039,user39@test.com,2025-02-02 18:16:46,United States,Florida,33101,standard_monthly,...,brand_awareness,firefox,linux,1170x2532,en-AU,EST,False,None,B002,c238d657
3,FEN-00063,2025-02-01 19:40:04,renew,FEN_C00034,user34@email.com,2025-02-12 21:59:18,GBR,Wales,CF10,premium_yearly,...,partner_q1,chrome,macos,1366x768,es-MX,CET,False,None,B002,6e5d1a78
4,FEN-00078,2025-02-01 21:28:15,renew,FEN_C00053,user53@email.com,2025-02-12 17:07:42,GBR,Wales,CF10,premium_yearly,...,latam_push,edge,windows,1366x768,en-GB,AEST,False,None,B002,b8f72a56
5,FEN-00029,2025-02-01 21:30:17,new,FEN_C00029,user29@email.com,2025-02-01 21:30:17,GBR,Wales,CF10,premium_monthly,...,eu_winter,chrome,android,1080x2340,ja-JP,PST,False,None,B001,0b8c31b4
6,FEN-00044,2025-02-02 10:15:43,new,FEN_C00043,user43@email.com,2025-02-02 10:15:43,USA,Washington,98101,premium_monthly,...,partner_q1,firefox,macos,2560x1440,de-DE,GMT,False,None,B001,47a03b2b
7,FEN-00058,2025-02-02 10:26:05,new,FEN_C00048,user48@test.com,2025-02-02 10:26:05,USA,New York,10001,standard_yearly,...,latam_push,edge,linux,1080x2340,de-DE,CST,False,None,B002,c967886a
8,FEN-00036,2025-02-02 16:44:26,new,FEN_C00036,user36@email.com,2025-02-02 16:44:26,USA,Florida,33101,standard_yearly,...,eu_winter,chrome,android,1080x2340,en-AU,AEST,False,None,B001,9a247786
9,FEN-00039,2025-02-02 18:16:46,new,FEN_C00039,user39@test.com,2025-02-02 18:16:46,USA,Florida,33101,standard_monthly,...,None,chrome,linux,1170x2532,pt-BR,GMT,False,None,B001,b5d5846f


In [11]:
exchange_df.printSchema()
exchange_df.limit(20).toPandas()

root
 |-- date: date (nullable = true)
 |-- currency: string (nullable = true)
 |-- rate_to_eur: double (nullable = true)



,date,currency,rate_to_eur
0,2025-02-01,USD,0.92
1,2025-03-01,USD,0.91
2,2025-04-01,USD,0.90
3,2025-05-01,USD,0.89
4,2025-06-01,USD,0.88
5,2025-07-01,USD,0.87
6,2025-08-01,USD,0.86
7,2025-09-01,USD,0.85
8,2025-10-01,USD,0.84
9,2025-11-01,USD,0.83


In [12]:
apfel_null_counts = apfel_df.select(
    [
        count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
        for c in apfel_df.columns
    ]
)
apfel_null_counts.toPandas()

,event_id,event_timestamp,event_type,customer_uuid,customer_email,customer_created_at,country_code,region,city,postal_code,...,currency,amount,tax_amount,discount_code,affiliate_id,device_type,app_version,session_id,internal_ref,processing_status
0,0,0,0,0,4,0,3,0,0,0,...,0,7,0,644,578,0,0,0,0,0


In [13]:
fenster_null_counts = fenster_df.select(
    [
        count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
        for c in fenster_df.columns
    ]
)
fenster_null_counts.toPandas()

,id,ts,type,cid,mail,signup_ts,ctry,state,zip,plan,...,utm_campaign,browser,os,screen_res,lang,tz,legacy_flag,migrated_from,batch_id,row_hash
0,0,0,0,0,0,0,1,0,0,0,...,125,0,0,0,0,0,0,1115,0,0


In [32]:
apfel_df.groupBy("event_type").count().orderBy(desc("count")).toPandas()

,event_type,count
0,SUBSCRIPTION_STARTED,748
1,SUBSCRIPTION_RENEWED,571
2,SUBSCRIPTION_CANCELLED,111
3,subscription_started,19
4,subscription_renewed,10
5,subscription_cancelled,1


In [15]:
fenster_df.groupBy("type").count().orderBy(desc("count")).toPandas()

+------+-----+
|  type|count|
+------+-----+
|   new|  620|
| renew|  443|
|cancel|   80|
|   NEW|   13|
| RENEW|    9|
+------+-----+



In [16]:
apfel_df.groupBy("subscription_type", "renewal_period").count().orderBy(
    desc("count")
).show()
apfel_df.groupBy("currency").count().show()

+-----------------+--------------+-----+
|subscription_type|renewal_period|count|
+-----------------+--------------+-----+
|         standard|       monthly|  383|
|          premium|        yearly|  378|
|         standard|        yearly|  372|
|          premium|       monthly|  327|
+-----------------+--------------+-----+

+--------+-----+
|currency|count|
+--------+-----+
|     GBP|    3|
|    Euro|    2|
|     eur|    2|
|     EUR| 1449|
|     USD|    4|
+--------+-----+



In [17]:
fenster_df.groupBy("plan").count().orderBy(desc("count")).show()
fenster_df.groupBy("ccy").count().show()

+----------------+-----+
|            plan|count|
+----------------+-----+
| premium_monthly|  301|
| standard_yearly|  301|
|standard_monthly|  289|
|  premium_yearly|  274|
+----------------+-----+

+---+-----+
|ccy|count|
+---+-----+
|USD| 1165|
+---+-----+



In [18]:
fenster_df.groupBy("ctry").count().orderBy(desc("count")).show(truncate=False)

+-------------+-----+
|ctry         |count|
+-------------+-----+
|GBR          |605  |
|USA          |547  |
|United States|9    |
|INVALID      |2    |
|XX           |1    |
|NULL         |1    |
+-------------+-----+



In [19]:
apfel_df.groupBy("country_code").count().orderBy(desc("count")).show(truncate=False)

+------------+-----+
|country_code|count|
+------------+-----+
|DE          |765  |
|GB          |692  |
|NULL        |3    |
+------------+-----+



In [21]:
print("Apfel: eventos por year-month")
apfel_df.withColumn(
    "ym", date_format(to_timestamp("event_timestamp"), "yyyy-MM")
).groupBy("ym").count().orderBy("ym").show(20)

print("Fenster: eventos por year-month")
fenster_df.withColumn("ym", date_format(to_timestamp("ts"), "yyyy-MM")).groupBy(
    "ym"
).count().orderBy("ym").show(20)

=== Apfel: eventos por year-month ===
+-------+-----+
|     ym|count|
+-------+-----+
|2025-02|  108|
|2025-03|   97|
|2025-04|  111|
|2025-05|  118|
|2025-06|  114|
|2025-07|  114|
|2025-08|  127|
|2025-09|  110|
|2025-10|  106|
|2025-11|  119|
|2025-12|  141|
|2026-01|  135|
|2026-02|   59|
|2026-03|    1|
+-------+-----+

=== Fenster: eventos por year-month ===
+-------+-----+
|     ym|count|
+-------+-----+
|2025-02|   90|
|2025-03|   93|
|2025-04|   94|
|2025-05|   84|
|2025-06|   97|
|2025-07|   80|
|2025-08|  103|
|2025-09|   89|
|2025-10|   95|
|2025-11|   87|
|2025-12|  107|
|2026-01|   93|
|2026-02|   51|
|2026-03|    2|
+-------+-----+



In [23]:
print("Apfel: amount nulo por event_type")
apfel_df.groupBy("event_type").agg(
    count("*").alias("total"),
    count(when(col("amount").isNull(), 1)).alias("amount_null"),
    count(when(col("amount") == 0, 1)).alias("amount_zero"),
).show()

print("Fenster: price nulo por type")
fenster_df.groupBy("type").agg(
    count("*").alias("total"),
    count(when(col("price").isNull(), 1)).alias("price_null"),
    count(when(col("price") == 0, 1)).alias("price_zero"),
).show()

=== Apfel: amount nulo por event_type ===
+--------------------+-----+-----------+-----------+
|          event_type|total|amount_null|amount_zero|
+--------------------+-----+-----------+-----------+
|SUBSCRIPTION_RENEWED|  571|          2|          0|
|subscription_started|   19|          0|          0|
|subscription_renewed|   10|          0|          0|
|SUBSCRIPTION_STARTED|  748|          4|          0|
|SUBSCRIPTION_CANC...|  111|          1|          0|
|subscription_canc...|    1|          0|          0|
+--------------------+-----+-----------+-----------+

=== Fenster: price nulo por type ===
+------+-----+----------+----------+
|  type|total|price_null|price_zero|
+------+-----+----------+----------+
|cancel|   80|         0|         0|
| RENEW|    9|         0|         0|
|   new|  620|         0|         0|
| renew|  443|         0|         0|
|   NEW|   13|         0|         0|
+------+-----+----------+----------+



In [25]:
fenster_df.describe("price", "tax").show()

+-------+------------------+-----------------+
|summary|             price|              tax|
+-------+------------------+-----------------+
|  count|              1165|             1165|
|   mean| 58.96682403433441|8.668206008583699|
| stddev|56.147865460287115|9.818526738338903|
|    min|              6.99|             0.56|
|    max|            149.99|             30.0|
+-------+------------------+-----------------+



In [26]:
# APFEL: valores unicos de device_type y processing_status
apfel_df.groupBy("device_type").count().orderBy(desc("count")).show()
apfel_df.groupBy("processing_status").count().orderBy(desc("count")).show()

+-----------+-----+
|device_type|count|
+-----------+-----+
|    android|  506|
|        ios|  496|
|        web|  458|
+-----------+-----+

+-----------------+-----+
|processing_status|count|
+-----------------+-----+
|        completed| 1418|
|          pending|   13|
|        COMPLETED|   11|
|           failed|    9|
|       processing|    9|
+-----------------+-----+



In [27]:
# FENSTER: legacy_flag y migrated_from
fenster_df.groupBy("legacy_flag").count().show()
fenster_df.groupBy("migrated_from").count().orderBy(desc("count")).show()

+-----------+-----+
|legacy_flag|count|
+-----------+-----+
|      false| 1115|
|       true|   50|
+-----------+-----+

+-------------+-----+
|migrated_from|count|
+-------------+-----+
|         NULL| 1115|
|    legacy_v1|   50|
+-------------+-----+



In [28]:
print("=== Apfel: emails nulos ===")
print(
    apfel_df.filter(
        col("customer_email").isNull() | (col("customer_email") == "")
    ).count()
)

print("=== Fenster: emails nulos ===")
print(fenster_df.filter(col("mail").isNull() | (col("mail") == "")).count())

=== Apfel: emails nulos ===


4
=== Fenster: emails nulos ===
0


In [29]:
# COBERTURA EXCHANGE RATES vs MESES EN FENSTER (USD)
fenster_months = (
    fenster_df.withColumn("ym", date_format(to_timestamp("ts"), "yyyy-MM"))
    .select("ym")
    .distinct()
    .orderBy("ym")
)

exchange_months = (
    exchange_df.withColumn("ym", date_format(to_timestamp("date"), "yyyy-MM"))
    .select("ym")
    .distinct()
    .orderBy("ym")
)

print("=== Meses en Fenster ===")
fenster_months.show(20)

print("=== Meses en Exchange Rates ===")
exchange_months.show(20)

missing = fenster_months.subtract(exchange_months)
print(f"=== Meses en Fenster SIN tasa de cambio: {missing.count()} ===")
missing.show()

=== Meses en Fenster ===
+-------+
|     ym|
+-------+
|2025-02|
|2025-03|
|2025-04|
|2025-05|
|2025-06|
|2025-07|
|2025-08|
|2025-09|
|2025-10|
|2025-11|
|2025-12|
|2026-01|
|2026-02|
|2026-03|
+-------+

=== Meses en Exchange Rates ===
+-------+
|     ym|
+-------+
|2025-02|
|2025-03|
|2025-04|
|2025-05|
|2025-06|
|2025-07|
|2025-08|
|2025-09|
|2025-10|
|2025-11|
|2025-12|
|2026-01|
|2026-02|
+-------+



=== Meses en Fenster SIN tasa de cambio: 1 ===


+-------+
|     ym|
+-------+
|2026-03|
+-------+



In [30]:
# RESUMEN DE HALLAZGOS
print("HALLAZGOS CLAVE:")
print("=" * 60)
print("1. Apfel: moneda siempre EUR, Fenster: siempre USD")
print("2. Apfel event_type: SUBSCRIPTION_STARTED/RENEWED/CANCELLED")
print("   Fenster type: new/renew/cancel")
print("3. Fenster 'plan' combina tipo+periodo (premium_monthly)")
print("   Apfel los tiene separados (subscription_type + renewal_period)")
print("4. Country codes inconsistentes en Fenster:")
print("   USA, United States, GBR, United Kingdom -> normalizar a ISO alpha-2")
print("5. Amount/price NULL en cancellations -> imputar 0")
print("6. Emails nulos en Apfel (customer_email) -> flag de calidad")
print("7. Fenster tiene legacy_flag y migrated_from -> datos migrados")
print("8. Timestamps: Apfel ISO 8601 (Z), Fenster 'yyyy-MM-dd HH:mm:ss'")
print("9. Posibles eventos con fechas futuras (>= 2026-03-13)")
print("10. Exchange rates cubren Feb 2025 - Feb 2026 (mensual)")
print("    -> verificar si hay meses de Fenster sin tasa")

HALLAZGOS CLAVE:
1. Apfel: moneda siempre EUR, Fenster: siempre USD
2. Apfel event_type: SUBSCRIPTION_STARTED/RENEWED/CANCELLED
   Fenster type: new/renew/cancel
3. Fenster 'plan' combina tipo+periodo (premium_monthly)
   Apfel los tiene separados (subscription_type + renewal_period)
4. Country codes inconsistentes en Fenster:
   USA, United States, GBR, United Kingdom -> normalizar a ISO alpha-2
5. Amount/price NULL en cancellations -> imputar 0
6. Emails nulos en Apfel (customer_email) -> flag de calidad
7. Fenster tiene legacy_flag y migrated_from -> datos migrados
8. Timestamps: Apfel ISO 8601 (Z), Fenster 'yyyy-MM-dd HH:mm:ss'
9. Posibles eventos con fechas futuras (>= 2026-03-13)
10. Exchange rates cubren Feb 2025 - Feb 2026 (mensual)
    -> verificar si hay meses de Fenster sin tasa


In [ ]:
spark.stop()
print("SparkSession detenida")